In [1]:
import csv
from collections import OrderedDict
from datetime import datetime

ordered_dict_sub = []

with open('pdi_data.csv', mode='r', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
        ordered_row = OrderedDict()
        for key, value in row.items():
            # Check if value looks like a date in dd.mm.yyyy format
            if isinstance(value, str) and '.' in value and len(value) >= 8:
                parts = value.split('.')
                if len(parts) == 3 and all(part.isdigit() for part in parts):
                    day = parts[0].zfill(2)
                    month = parts[1].zfill(2)
                    year = parts[2]
                    value = f"{day}/{month}/{year}"  # Format: dd/mm/yyyy
            ordered_row[key] = value
        ordered_dict_sub.append(ordered_row)

# Sort by date column
DATE_COLUMN = 'Date'  # Change if your column header is different

ordered_dict_sub.sort(key=lambda x: datetime.strptime(x[DATE_COLUMN], "%d/%m/%Y"))

# Output
'''
print(f"Total rows: {len(ordered_dict_sub)}")
for item in ordered_dict_sub:
    print(item)
'''

'\nprint(f"Total rows: {len(ordered_dict_sub)}")\nfor item in ordered_dict_sub:\n    print(item)\n'

In [2]:
import pandas as pd
#from collections import OrderedDict

# Read all sheets from the Excel file into a dictionary
xls = pd.read_excel('MASTER_DF_updated.xlsx', sheet_name=None)

# View all available sheet names
sheet_names = list(xls.keys())
print("Sheets found:", sheet_names)

# Dictionary to store OrderedDicts for each trial
ordered_data_by_sheet = OrderedDict()

# Access each sheet individually and convert rows to OrderedDict
df_trial1 = xls.get('Trial1')
df_trial2 = xls.get('Trial2')
df_trial3 = xls.get('Trial3')
df_trial4 = xls.get('Trial4')
df_trial5 = xls.get('Trial5')
df_trial6 = xls.get('Trial6')
df_trial7 = xls.get('Trial7')
df_trial8 = xls.get('Trial8')
df_trial9 = xls.get('Trial9')
df_trial10 = xls.get('Trial10')
df_trial11 = xls.get('Trial11')

# Manually list trials you want to convert
trial_dfs = {
    'Trial1': df_trial1,
    'Trial2': df_trial2,
    'Trial3': df_trial3,
    'Trial4': df_trial4,
    'Trial5': df_trial5,
    'Trial6': df_trial6,
    'Trial7': df_trial7,
    'Trial8': df_trial8,
    'Trial9': df_trial9,
    'Trial10': df_trial10,
    'Trial11': df_trial11
}

# Convert each DataFrame row into OrderedDict and store
for sheet_name, df in trial_dfs.items():
    if df is not None:
        records = [OrderedDict(row.items()) for _, row in df.iterrows()]
        ordered_data_by_sheet[sheet_name] = records


# Optional: print a preview
for sheet, records in ordered_data_by_sheet.items():
    print(f"\nSheet: {sheet}")
    for i, row in enumerate(records[:2]):  # preview first 2 rows per sheet
        print(f" Row {i+1}: {row}")

Sheets found: ['Trial1', 'Trial2', 'Trial3', 'Trial4', 'Trial5', 'Trial6', 'Trial7', 'Trial8', 'Trial9', 'Trial10', 'Trial11', 'Sheet11']

Sheet: Trial1
 Row 1: OrderedDict({'PlotID': 1, 'Replication': 'R1', 'Treatment': 'T1', 'Name of plot': 'R1T1', 'Date': '28/08/2023', 'Plant height': 51.16, 'Leaf length': 45.55, 'Psudostem Length': 6.0, 'Number of leaves': 6.5, 'Psuetostem width': 5.3, 'Leaf width': 3.31, 'PDI Stemphylium Blight': 0, 'PDI Purple blotch': 0, 'PDI AN': 0, 'Thrips': 43.3, 'new_thrips': 43.3, 'DAT': 48})
 Row 2: OrderedDict({'PlotID': 2, 'Replication': 'R1', 'Treatment': 'T2', 'Name of plot': 'R1T2', 'Date': '28/08/2023', 'Plant height': 51.11, 'Leaf length': 43.01, 'Psudostem Length': 6.62, 'Number of leaves': 6.6, 'Psuetostem width': 11.02, 'Leaf width': 6.36, 'PDI Stemphylium Blight': 2, 'PDI Purple blotch': 0, 'PDI AN': 0, 'Thrips': 4.6, 'new_thrips': 4.6, 'DAT': 48})

Sheet: Trial2
 Row 1: OrderedDict({'PlotID': 1, 'Replication': 'R1', 'Treatment': 'T1', 'Name of 

In [3]:
from copy import deepcopy
# Loop through each row in CSV (ordered_dict_sub)
# New container for updated Excel data
updated_main_sheet = OrderedDict()
for row in ordered_dict_sub:
    # Step 1: Access and clean 'Trial name'
    trial_row = row.get('\ufeffTrial name') or row.get('Trial name')
    #print ("sub", row)
    if not trial_row:
        print(f"Missing trial name in row: {row}")
        continue
    #print (trial_row)

    # Step 2: Normalize 'Trial 5' → 'Trial5'
    trial_key = trial_row.replace(" ", "")

    #print (trial_key)

     # Step 3: Match with ordered_data_by_sheet
    if trial_key in ordered_data_by_sheet:
        trial_data = ordered_data_by_sheet[trial_key]
        # Prepare to hold enriched rows
        if trial_key not in updated_main_sheet:
            updated_main_sheet[trial_key] = []

        # Step 4: Match rows inside this sheet by Date, Treatment, Replication
        for main_row in trial_data:
            updated_row = deepcopy(main_row)

            # Match condition
            if (
                str(main_row.get('Treatment')) == str(row.get('Treatment')) and
                str(main_row.get('Replication')) == str(row.get('Replication')) and
                str(main_row.get('Date')) == str(row.get('Date'))
            ):
                #print ("sub: ", row, "\n", "main: ", main_row, "\n")
                updated_row = deepcopy(main_row)  # Step 6: Copy main_row

                # Step 7: Add new values from row
                updated_row['PDI_SB_new'] = str(row.get('PDI_SB'))
                updated_row['PDI_PB_new'] = str(row.get('PDI_PB'))
                updated_row['PDI_AN_new'] = str(row.get('PDI_AN'))
                #updated_row['Trial name'] = trial_key

                #print ("updated_row: ", updated_row, "\n\n")

                # Step 8: Add to final dictionary
                updated_main_sheet[trial_key].append(updated_row)
                #matched = True
                #break
            #break
    #print ("master sheet", trial_data[:2])

In [4]:
print(f"Total rows: {len(updated_main_sheet)}")
for item in updated_main_sheet:
    print(item)

Total rows: 5
Trial5
Trial6
Trial7
Trial8
Trial9


In [5]:
import pandas as pd

# Save to a new Excel file
output_filename = "updated_trials_no_dates.xlsx"

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    print(f"Total sheets: {len(updated_main_sheet)}")
    
    for sheet_name, row_list in updated_main_sheet.items():
        print(f"Writing sheet: {sheet_name} with {len(row_list)} rows")

        # Convert list of dicts to DataFrame
        df = pd.DataFrame(row_list)

        # Excel sheet names must be <=31 characters and cannot contain certain characters
        safe_sheet_name = sheet_name[:31].replace('/', '_').replace('\\', '_').replace('*', '_').replace('[', '_').replace(']', '_')

        # Write to sheet
        df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

Total sheets: 5
Writing sheet: Trial5 with 30 rows
Writing sheet: Trial6 with 45 rows
Writing sheet: Trial7 with 15 rows
Writing sheet: Trial8 with 45 rows
Writing sheet: Trial9 with 75 rows
